<a href="https://colab.research.google.com/github/Anshgupta2906/AI_agent_from_scratch/blob/main/02_tool_pattern/tool_pattern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai requests -q

In [2]:
import os
import json
import re
import time
import requests
import subprocess
from openai import OpenAI
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
FREE_MODEL = "openrouter/free"
TOOL_MODEL = "openrouter/free"

In [3]:
def calculator(expression):
    result = eval(expression)
    return result

# test
print(calculator("2 + 2"))
print(calculator("10 * 5"))
print(calculator("(100 - 20) / 4"))

4
50
20.0


In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Use this to perform mathematical calculations.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A valid Python math expression e.g. '2 + 2' or '10 * 5'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]


In [10]:

Response=client.chat.completions.create(
    model=FREE_MODEL,
    messages=[
        {"role":"user","content":'what is the square of 12'}
    ],
    tools=tools,
    tool_choice="auto"

)
print(Response)

ChatCompletion(id='gen-1775223429-a4hlIxEYA5UqTnh6FsAp', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call-0ee55c81-9627-4740-9f4e-87e01641df7f', function=Function(arguments='{"expression":"12 ** 2"}', name='calculator'), type='function', index=0)], reasoning=None), native_finish_reason='tool_calls')], created=1775223429, model='arcee-ai/trinity-large-preview:free', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=22, prompt_tokens=166, total_tokens=188, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_token

In [12]:
tool_call=Response.choices[0].message.tool_calls[0]
arguments=json.loads(tool_call.function.arguments)
expression=arguments['expression']
result=calculator(expression)
print(result)

144


In [14]:
messages= [
    {"role":"user","content":"what is the square of 12"},
    Response.choices[0].message,
    {
        "role":"tool",
        "tool_call_id" :tool_call.id,
        "content":str(result)
    }
]
final_response=client.chat.completions.create(
    model=FREE_MODEL,
    messages=messages
)

print(final_response.choices[0].message.content)

The square of 12 is **144**.  
(12 × 12 = 144)

